# Day 4 v2 — Model 05: AITeamVN/Vietnamese_Embedding + DNN (Hướng B)

**Architecture:** `AITeamVN/Vietnamese_Embedding` (frozen, 568M params) → 1024-dim dense embedding → PriceDNN head (6 ResidualBlocks)

**Tại sao AITeamVN:** VN-MTEB score cao nhất (63.34), classification score 69.06 — phù hợp price prediction theo category. BGE-M3 base, 512-token limit.

**Note:** Model nặng 568M. Dùng `encode_batch_size=64` để tránh OOM khi encode. input_size=1024 (auto-detected bởi SentTransRunner.setup).

**Target:** MAE < 70k VND

**Dataset:** `SeanSunny/items_tv_v9` (train=269K, val=3926, test=3872)

## vast.ai Setup (chỉ chạy lần đầu)

```bash
pip install uv
uv sync
```

Restart kernel sau khi sync xong.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import json
import torch

from pricer_vi_2.items import Item
from pricer_vi_2.evaluator import evaluate, plot_training_history
import pricer_vi_2.senttrans_model as sm

# Override encoder BEFORE creating runner
sm.ENCODER_NAME = "AITeamVN/Vietnamese_Embedding"

from pricer_vi_2.senttrans_model import SentTransRunner

print(f"Encoder: {sm.ENCODER_NAME}")
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.9.0+cu128).


Encoder: AITeamVN/Vietnamese_Embedding
CUDA: True
GPU: NVIDIA GeForce RTX 3090 Ti
VRAM: 25.3 GB


## 1. Load Data

In [2]:
train, val, test = Item.from_hub("SeanSunny/items_tv_v9")
print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")

Train: 269,112 | Val: 3,926 | Test: 3,872


## 2. Pre-compute Embeddings

AITeamVN/Vietnamese_Embedding (568M, BGE-M3 base) → 1024-dim embeddings.
`encode_batch_size=64` để tránh OOM (model nặng hơn e5-small 5x).
Lần đầu: ~15-20 phút trên GPU. Lần sau: load từ cache pkl.

> Nếu vẫn OOM: đổi `encode_batch_size=32`.

In [3]:
runner = SentTransRunner(train, val)

cache_path = Path("cache/aitvn_embeddings.pkl")
runner.encode_and_cache(cache_path=cache_path, encode_batch_size=64)

Loading encoder: AITeamVN/Vietnamese_Embedding


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

Encoding train embeddings (269K) — ~10 min on GPU...


Batches:   0%|          | 0/4205 [00:00<?, ?it/s]

Encoding val embeddings (3926)...


Batches:   0%|          | 0/62 [00:00<?, ?it/s]

Embeddings cached to cache/aitvn_embeddings.pkl
Train: torch.Size([269112, 1024]) | Val: torch.Size([3926, 1024])


## 3. Setup Model

DNN head: 6 ResidualBlocks. input_size=1024 (auto-detected từ embedding dim).
`batch_size=128` cho DNN training (1024-dim activations nặng hơn 384-dim).

In [4]:
runner.setup(batch_size=128, num_blocks=6)

SentTrans DNN head: 205,684,737 trainable params | embedding_dim=1024
Using cuda


## 4. Train

Max 15 epochs, early stopping patience=3. Val feedback dùng val[:1000] mỗi epoch.

In [5]:
history = runner.train(epochs=15, patience=3)

Epoch 1/15:   0%|          | 0/2103 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PAR

Epoch 1/15 | train_loss=0.8546 | val_loss=0.5206 | val_mae=124.79k | lr=0.000989
  ** best val_mae=124.79k


Epoch 2/15:   0%|          | 0/2103 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PAR

Epoch 2/15 | train_loss=0.4190 | val_loss=0.4372 | val_mae=108.40k | lr=0.000957
  ** best val_mae=108.40k


Epoch 3/15:   0%|          | 0/2103 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PAR

...

Epoch 13/15 | train_loss=0.0783 | val_loss=0.3663 | val_mae=90.41k | lr=0.000043


Epoch 14/15:   0%|          | 0/2103 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PA

Epoch 14/15 | train_loss=0.0730 | val_loss=0.3664 | val_mae=89.85k | lr=0.000011


Epoch 15/15:   0%|          | 0/2103 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PA

Epoch 15/15 | train_loss=0.0705 | val_loss=0.3669 | val_mae=90.01k | lr=0.000000
  Early stopping. Best val_mae=89.69k


## 5. Training History

In [6]:
plot_training_history(history, title="AITeamVN Vietnamese_Embedding + DNN")

## 6. Save Weights + Val Predictions + Test Predictions

In [7]:
Path("weights").mkdir(exist_ok=True)
runner.save("weights/aitvn_dnn.pth")
print("Saved weights/aitvn_dnn.pth")

Path("val_predictions").mkdir(exist_ok=True)

print("Running val predictions (3926 samples)...")
val_preds = runner.val_predictions()
with open("val_predictions/aitvn_val.json", "w") as f:
    json.dump(val_preds, f)

print("Encoding + predicting test set (3872 samples)...")
test_preds = runner.test_predictions(test, encode_batch_size=64)
with open("val_predictions/aitvn_test.json", "w") as f:
    json.dump(test_preds, f)

print(f"Val: {len(val_preds)} | Test: {len(test_preds)}")

Saved weights/aitvn_dnn.pth
Running val predictions (3926 samples)...


Encoding + predicting test set (3872 samples)...


Batches:   0%|          | 0/61 [00:00<?, ?it/s]

Val: 3926 | Test: 3872


## 7. Evaluate on 200 Test Samples

In [8]:
def aitvn_pricer(item):
    return runner.inference(item)

results = evaluate(aitvn_pricer, test)
print(f"MAE: {results['mae']:.1f}k VND | MSE: {results['mse']:,.0f} | R2: {results['r2']:.1f}%")

  0%|          | 0/200 [00:00<?, ?it/s]

298 57 21 1 40 437 87 48 54 12 19 25 79 36 109 113 0 34 122 12 195 5 2 13 35 5 113 46 522 17 36 82 4 4 16 69 43 20 107 4 93 26 69 3 9 364 6 6 102 43 6 59 139 472 87 100 93 73 11 139 156 134 10 54 58 99 1 100 273 7 64 44 2 57 6 64 70 16 9 66 157 491 171 213 38 109 12 5 102 11 10 2 23 365 3 7 52 12 39 17 91 31 59 9 130 28 26 64 6 3 45 31 186 170 70 20 7 9 29 11 184 43 2 16 23 64 7 8 137 13 38 11 238 1 101 10 213 268 281 33 32 57 102 2 195 11 117 61 29 1 274 8 54 132 8 236 4 147 2 27 62 26 11 14 1 164 21 206 21 203 96 42 35 111 5 55 3 2 44 33 389 9 26 8 25 123 11 426 462 5 149 36 1 16 28 47 3 102 0 7 


MAE: 76.1k VND | MSE: 16,254 | R2: 71.7%


## 8. Sanity Check — Load Roundtrip

In [9]:
sample = test[0]
pred_original = runner.inference(sample)
print(f"Product: {sample.title[:60]}")
print(f"Actual:  {sample.price:.1f}k VND")
print(f"Predict: {pred_original:.1f}k VND")
print(f"Error:   {abs(pred_original - sample.price):.1f}k VND")

runner.load("weights/aitvn_dnn.pth")
pred_loaded = runner.inference(sample)
diff = abs(pred_original - pred_loaded)
assert diff < 0.01, f"Load mismatch: before={pred_original:.2f} after={pred_loaded:.2f}"
print(f"\nLoad roundtrip PASSED. Diff: {diff:.4f}k")

Product: Thùng lưu trữ, hộp đựng đồ đa năng bằng nhựa PP cao cấp 30L 
Actual:  479.0k VND
Predict: 181.3k VND
Error:   297.7k VND

Load roundtrip PASSED. Diff: 0.0000k
